In [ ]:
# CWBagging versi clean (by: ChatGPT)

"""
CWBagging: Class-Weighted Bagging for Multiclass Imbalanced Learning

This script evaluates the CWBagging ensemble method using multiple
datasets with repeated Stratified K-Fold cross validation.

Main characteristics:
- Custom bootstrap sampling
- Random oversampling during early ensemble iterations
- Class-weighted soft voting
- Evaluation using AvAcc, G-Mean, and MAUC
- Runtime measurement

Author: Muhammad Fachrie
"""

import numpy as np
import pandas as pd
import time
import warnings
import random

from collections import Counter

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.preprocessing import LabelEncoder

from imblearn.over_sampling import RandomOverSampler

warnings.filterwarnings("ignore")


# =========================================================
# UTILITY FUNCTIONS
# =========================================================

def calculate_class_weights(y):
    """
    Compute inverse-frequency class weights.

    Parameters
    ----------
    y : array-like
        Class labels

    Returns
    -------
    dict
        Mapping class -> weight
    """
    n_samples = len(y)
    n_classes = len(set(y))
    counts = Counter(y)

    weights = {
        cls: n_samples / (n_classes * count)
        for cls, count in counts.items()
    }

    return weights


# =========================================================
# CUSTOM BOOTSTRAP SAMPLING
# =========================================================

def custom_bootstrap(X, y):
    """
    Generate a bootstrap sample by drawing samples
    independently for each class.

    The number of samples drawn per class is randomly
    selected between 2 and half of the largest class size.

    Returns
    -------
    X_boot, y_boot
    """

    unique_classes = np.unique(y)

    class_counts = {
        cls: np.sum(y == cls)
        for cls in unique_classes
    }

    largest_class_size = max(class_counts.values())
    threshold = largest_class_size // 2

    m = np.random.randint(2, threshold) if threshold > 2 else 2

    indices = []

    for cls in unique_classes:

        class_idx = np.where(y == cls)[0]

        if len(class_idx) == 0:
            continue

        sample_size = min(m, len(class_idx))

        sampled = np.random.choice(
            class_idx,
            size=sample_size,
            replace=True
        )

        indices.extend(sampled)

    indices = np.array(indices)

    return X[indices], y[indices]


# =========================================================
# CLASS-WEIGHTED SOFT VOTING
# =========================================================

def soft_voting_with_weights(models, X, class_weight_list, threshold=0):
    """
    Aggregate predictions from ensemble members using
    confidence-weighted voting.

    Each prediction is weighted by:
        confidence * class_weight

    Parameters
    ----------
    models : list
        Trained classifiers

    X : array
        Test samples

    class_weight_list : list
        Class weight dictionaries

    threshold : float
        Minimum confidence threshold

    Returns
    -------
    probabilities : ndarray
        Final probability distribution
    """

    n_samples = X.shape[0]
    n_classes = models[0].n_classes_

    weighted_votes = np.zeros((n_samples, n_classes))

    for model, weights in zip(models, class_weight_list):

        proba = model.predict_proba(X)

        preds = np.argmax(proba, axis=1)
        conf = np.max(proba, axis=1)

        for i in range(n_samples):

            cls = preds[i]

            if conf[i] >= threshold and cls in weights:
                weighted_votes[i, cls] += conf[i] * weights[cls]

    row_sum = weighted_votes.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1

    return weighted_votes / row_sum


# =========================================================
# PERFORMANCE METRICS
# =========================================================

def calculate_metrics(y_true, y_pred, n_classes):
    """
    Compute evaluation metrics for multiclass classification.

    Metrics:
    - Average Accuracy (AvAcc)
    - Geometric Mean (G-Mean)
    """

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))

    with np.errstate(divide='ignore', invalid='ignore'):

        recall = np.diag(cm) / np.sum(cm, axis=1)
        recall = np.nan_to_num(recall)

    avacc = np.mean(recall)

    gmean = (
        np.prod(recall) ** (1 / n_classes)
        if np.all(recall > 0)
        else 0.0
    )

    return avacc, gmean


# =========================================================
# EXPERIMENT SETTINGS
# =========================================================

#-----------------------------------
# for 5-fold cross validation, use:
# ---> ecoli-5fold.csv, 
# ---> automobile-5fold.csv,
# ---> lymphography-5fold.csv,
# ---> shuttle-5fold.csv
#-----------------------------------
datasets = [
    "winequality-red.csv",
    "yeast.csv",
    "glass.csv",
    "ecoli.csv",
    "dermatology.csv",
    "solar-flare.csv",
    "automobile.csv",
    "contraceptive.csv",
    "balance-scale.csv",
    "led7digit.csv",
    "lymphography.csv",
    "hayes-roth.csv",
    "cleveland.csv",
    "new-thyroid.csv",
    "page-blocks.csv",
    "wine.csv",
    "thyroid.csv",
    "zoo.csv",
    "car.csv",
    "shuttle.csv"
]

#------------------------------
# for 2-fold cross validation
#------------------------------
random_seeds = [123, 1000, 14, 0, 65, 456, 321, 11, 56, 100, 999, 431, 648, 101, 111, 333, 5000, 2, 665, 9, 518, 222, 5, 20, 1]

#------------------------------
# for 5-fold cross validation
#------------------------------
# random_seeds = [123, 1000, 14, 0, 65, 456, 321, 11, 56, 100]

n_repeats = 25
n_folds = 2
n_estimators = 100


# =========================================================
# MAIN EXPERIMENT LOOP
# =========================================================

results = []

for dataset in datasets:

    print(f"\nProcessing dataset: {dataset}")

    df = pd.read_csv(dataset)

    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values

    y = LabelEncoder().fit_transform(y)

    n_classes = len(np.unique(y))

    avacc_scores = []
    gmean_scores = []
    mauc_scores = []

    train_times = []
    test_times = []

    for r in range(n_repeats):

        skf = StratifiedKFold(
            n_splits=n_folds,
            shuffle=True,
            random_state=random_seeds[r]
        )

        for train_idx, test_idx in skf.split(X, y):

            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            models = []
            weight_list = []

            fold_train_time = 0

            # ---------------------------------------------------------
            # Calculating Class Weights within the entire training set
            # ---------------------------------------------------------
            class_weights_train = calculate_class_weights(y_train)

            # -----------------------------
            # Training Phase
            # -----------------------------

            for est in range(n_estimators):

                X_bs, y_bs = custom_bootstrap(X_train, y_train)

                if len(np.unique(y_bs)) < n_classes:
                    continue

                # -------------------------------------------------------
                # Calculating Class Weights within the current bootstrap
                # -------------------------------------------------------
                weights = calculate_class_weights(y_bs)

                common = weights.keys() & class_weights_train.keys()

                # -----------------------------
                # Weight Capping
                # -----------------------------
                weights = {
                    c: min(weights[c], class_weights_train[c])
                    for c in common
                }

                if est < n_estimators * 0.5:
                    
                    # -----------------------------------------
                    # Oversampling on the first 50% bootstrap
                    # -----------------------------------------
                    ros = RandomOverSampler()
                    X_bs, y_bs = ros.fit_resample(X_bs, y_bs)

                model = DecisionTreeClassifier()

                start_fit = time.time()

                model.fit(X_bs, y_bs)

                end_fit = time.time()

                fold_train_time += end_fit - start_fit

                models.append(model)
                weight_list.append(weights)

            train_times.append(fold_train_time)

            # -----------------------------
            # Testing Phase
            # -----------------------------

            start_test = time.time()

            proba = soft_voting_with_weights(
                models,
                X_test,
                weight_list
            )

            pred = np.argmax(proba, axis=1)

            end_test = time.time()

            test_times.append(
                (end_test - start_test) / len(y_test)
            )

            # -----------------------------
            # Evaluation
            # -----------------------------

            avacc, gmean = calculate_metrics(
                y_test,
                pred,
                n_classes
            )

            avacc_scores.append(avacc)
            gmean_scores.append(gmean)

    results.append({

        "Dataset": dataset,

        "AvAcc_mean": np.mean(avacc_scores),
        "AvAcc_std": np.std(avacc_scores),

        "GMean_mean": np.mean(gmean_scores),
        "GMean_std": np.std(gmean_scores),

        "TrainTime_mean": np.mean(train_times),
        "TrainTime_std": np.std(train_times),

        "TestTime_mean": np.mean(test_times),
        "TestTime_std": np.std(test_times)

    })

    print(f"AvAcc : {np.mean(avacc_scores):.4f} ± {np.std(avacc_scores):.4f}")
    print(f"GMean : {np.mean(gmean_scores):.4f} ± {np.std(gmean_scores):.4f}")

# =========================================================
# SAVE RESULTS
# =========================================================

results_df = pd.DataFrame(results)

output_file = "CWBagging_results.csv"

results_df.to_csv(output_file, index=False)

print("\nResults saved to:", output_file)


Processing dataset: winequality-red.csv
AvAcc : 0.4048 ± 0.0401
GMean : 0.3485 ± 0.1227

Processing dataset: yeast.csv
AvAcc : 0.5614 ± 0.0197
GMean : 0.4903 ± 0.0751

Processing dataset: glass.csv
AvAcc : 0.7431 ± 0.0356
GMean : 0.7026 ± 0.0405

Processing dataset: ecoli.csv
AvAcc : 0.5439 ± 0.0486
GMean : 0.0000 ± 0.0000

Processing dataset: dermatology.csv
AvAcc : 0.9642 ± 0.0148
GMean : 0.9627 ± 0.0163

Processing dataset: solar-flare.csv
AvAcc : 0.6405 ± 0.0209
GMean : 0.5589 ± 0.0276

Processing dataset: automobile.csv
AvAcc : 0.6880 ± 0.0635
GMean : 0.6500 ± 0.0785

Processing dataset: contraceptive.csv
AvAcc : 0.5312 ± 0.0131
GMean : 0.5298 ± 0.0133

Processing dataset: balance-scale.csv
AvAcc : 0.6482 ± 0.0383
GMean : 0.6359 ± 0.0503

Processing dataset: led7digit.csv


In [5]:
# Kode CWBagging dengan variasi base classifier + MAUC + Waktu

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
import random
import warnings
import time

warnings.filterwarnings('ignore')

# --- SEMUA FUNGSI PEMBANTU ---

def calculate_class_weights(y):
    n_samples = len(y)
    n_classes = len(set(y))
    class_counts = Counter(y)
    class_weights = {cls: n_samples / (n_classes * count) for cls, count in class_counts.items()}
    
    # skenario class_weights = 1, untuk cek ablation tanpa class_weights
    # class_weights = {cls: n_samples / n_samples for cls, count in class_counts.items()}

    return class_weights

def duplicate_minority_classes(X, y, min_count=5):
    unique_classes, counts = np.unique(y, return_counts=True)
    new_X, new_y = X.copy(), y.copy()
    min_required = max(min_count, 5) # Pastikan minimal 5 untuk 5-fold CV

    for cls, count in zip(unique_classes, counts):
        if count < min_required:
            class_indices = np.where(y == cls)[0]
            n_duplicates = min_required - count
            if len(class_indices) > 0:
                # np.random.seed(42)
                duplicates = np.random.choice(class_indices, size=n_duplicates, replace=True)
                new_X = np.vstack([new_X, X[duplicates]])
                new_y = np.hstack([new_y, y[duplicates]])
    return new_X, new_y

# def calculate_metrics(y_true, y_pred, n_classes):
#     cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
#     recall_per_class = np.diag(cm) / np.sum(cm, axis=1, where=np.sum(cm, axis=1)!=0)
#     avacc = np.mean(recall_per_class)
#     mgm = np.prod(recall_per_class) ** (1 / n_classes) if n_classes > 0 else 0.0
#     cba = np.mean([
#         cm[i, i] / max(np.sum(cm[i, :]), np.sum(cm[:, i]))
#         for i in range(n_classes) if max(np.sum(cm[i, :]), np.sum(cm[:, i])) > 0
#     ]) if n_classes > 0 else 0.0
#     macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
#     return avacc, mgm, cba, macro_f1, recall_per_class

def calculate_metrics(y_true, y_pred, y_proba, n_classes):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
    with np.errstate(divide='ignore', invalid='ignore'):
        recall_per_class = np.diag(cm) / np.sum(cm, axis=1)
        recall_per_class = np.nan_to_num(recall_per_class, nan=0.0)
    avacc = np.mean(recall_per_class)
    gmean = np.prod(recall_per_class)**(1/n_classes) if np.all(recall_per_class > 0) else 0.0
    try:
        mauc = roc_auc_score(pd.get_dummies(y_true), y_proba, average='macro', multi_class='ovr')
    except:
        mauc = 0.0
    return avacc, gmean, mauc

def custom_bootstrap(X, y):
    unique_classes = np.unique(y)
    class_counts = {cls: len(np.where(y == cls)[0]) for cls in unique_classes}
    largest_class_size = class_counts[max(class_counts, key=class_counts.get)] if class_counts else 1
    smallest_class_size = class_counts[min(class_counts, key=class_counts.get)] if class_counts else 1
    avg_class_size = int(np.mean(list(class_counts.values()))) if class_counts else 1
    median_class_size = int(np.median(list(class_counts.values()))) if class_counts else 1
    
    # threshold = (largest_class_size + smallest_class_size) // 2
    threshold = (largest_class_size) // 2
    # threshold = avg_class_size
    # threshold = largest_class_size
    # threshold = median_class_size
    # threshold = max((largest_class_size // 2), avg_class_size)
    
    # if threshold <= 2: threshold = max(3, largest_class_size) # Pastikan threshold valid
    m = np.random.randint(2, threshold) if threshold > 2 else 2
    # m = np.random.randint(smallest_class_size, threshold) if threshold > 2 else 2
    # m = np.random.randint(smallest_class_size, threshold)
    # m = threshold

    indices = np.array([], dtype=int)
    for cls in unique_classes:
        class_indices = np.where(y == cls)[0]
        class_size = len(class_indices)
        if not class_indices.any(): continue # Skip jika tidak ada index

        # Tentukan ukuran sample, pastikan tidak 0
        current_m = m if class_size > m else class_size
        if current_m == 0: continue # Skip jika ukuran sample 0

        sampled_indices = np.random.choice(class_indices, size=current_m, replace=True)
        indices = np.concatenate([indices, sampled_indices])
    return X[indices], y[indices]

def soft_voting_with_weights(models, X, class_weight_list, return_proba=False):
#     if not models: # Handle jika list model kosong
#         # Kembali ke prediksi random atau mayoritas? Kita return proba uniform.
#         n_samples = X.shape[0]
#         # Perlu tahu n_classes. Ambil dari class_weight_list jika ada?
#         n_classes_est = 0
#         if class_weight_list:
#              n_classes_est = len(class_weight_list[0].keys()) # Asumsi dict tidak kosong
#         if n_classes_est == 0: n_classes_est = 2 # Default?

#         if return_proba:
#              return np.ones((n_samples, n_classes_est)) / n_classes_est
#         else:
#              return np.random.randint(0, n_classes_est, size=n_samples)


    n_samples = X.shape[0]
    n_classes = models[0].n_classes_
    weighted_predictions = np.zeros((n_samples, n_classes))

    for i, model in enumerate(models):
        probas = model.predict_proba(X)
        # Handle jika probas tidak lengkap (model tidak melihat semua kelas saat fit)
#         if probas.shape[1] < n_classes:
#             temp_probas = np.zeros((n_samples, n_classes))
#             # Asumsi model.classes_ menyimpan kelas yg dilihat
#             try:
#                 present_classes_idx = [np.where(model.classes_ == c)[0][0] for c in model.classes_ if c < n_classes]
#                 temp_probas[:, model.classes_] = probas[:, present_classes_idx]
#             except: # Fallback jika model.classes_ tidak sesuai
#                  temp_probas[:, :probas.shape[1]] = probas # Asumsi kelas 0..k-1
#             probas = temp_probas

        preds = np.argmax(probas, axis=1)
        confidences = np.max(probas, axis=1)
        class_weights = class_weight_list[i]
        
        for j in range(n_samples):
            pred_class = preds[j]
            if pred_class in class_weights:
                if confidences[j] >= threshold:
                    weighted_predictions[j, pred_class] += confidences[j] * class_weights[pred_class]

    if return_proba:
        row_sums = weighted_predictions.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        return weighted_predictions / row_sums

    final_predictions = np.argmax(weighted_predictions, axis=1)
    return final_predictions

def calculate_q_statistic(y_true, predictions):
    # ... (Fungsi ini tidak diubah, tapi pastikan `predictions` adalah list of arrays) ...
    if not predictions or len(predictions) < 2: return 0.0 # Handle jika tidak cukup model
    n_models = len(predictions)
    q_values = []
    # Pastikan y_true adalah 1D array
    y_true = np.array(y_true).flatten()

    for i in range(n_models):
        for j in range(i + 1, n_models):
            # Pastikan pred_i dan pred_j punya shape sama dengan y_true
            pred_i = np.array(predictions[i]).flatten()
            pred_j = np.array(predictions[j]).flatten()
            if pred_i.shape != y_true.shape or pred_j.shape != y_true.shape:
                # print(f"Warning: Shape mismatch in Q-statistic. y_true: {y_true.shape}, pred_i: {pred_i.shape}, pred_j: {pred_j.shape}")
                continue # Lewati pasangan ini jika shape tidak cocok

            correct_i = (pred_i == y_true)
            correct_j = (pred_j == y_true)

            N11 = np.sum(correct_i & correct_j)
            N00 = np.sum(~correct_i & ~correct_j)
            N10 = np.sum(correct_i & ~correct_j)
            N01 = np.sum(~correct_i & correct_j)

            denominator = (N11 * N00 + N10 * N01)
            q = (N11 * N00 - N10 * N01) / denominator if denominator != 0 else 0
            q_values.append(q)

    return np.mean(q_values) if q_values else 0.0

# --- PENGATURAN EKSPERIMEN ---

base_classifiers = {
    'DecisionTree': DecisionTreeClassifier(),
}
# Dataset untuk 2-fold
# list_datasets = ['winequality-red.csv', 'yeast.csv', 'glass.csv', 'ecoli.csv', 'dermatology.csv', 'solar-flare.csv', 'automobile.csv',
#                  'contraceptive.csv', 'balance-scale.csv', 'led7digit.csv', 'lymphography.csv', 'hayes-roth.csv',
#                  'cleveland.csv', 'new-thyroid.csv', 'page-blocks.csv', 'wine.csv', 'thyroid.csv', 'zoo.csv', 'car.csv', 'shuttle.csv']

# Dataset untuk 5-fold
list_datasets = ['winequality-red.csv', 'yeast.csv', 'glass.csv', 'ecoli 5-fold.csv', 'dermatology.csv', 'solar-flare.csv',
                 'automobile 5-fold.csv', 'contraceptive.csv', 'balance-scale.csv', 'led7digit.csv', 'lymphography 5-fold.csv',
                 'hayes-roth.csv', 'cleveland.csv', 'new-thyroid.csv', 'page-blocks.csv', 'wine.csv', 'thyroid.csv',
                 'zoo.csv', 'car.csv', 'shuttle 5-fold.csv']

# list_datasets = ['cleveland.csv', 'cleveland.csv', 'cleveland.csv', 'cleveland.csv', 'cleveland.csv', 'cleveland.csv']

summary_results = []
threshold = 0
# list_seed = [123, 1000, 14, 0, 65, 456, 321, 11, 56, 100, 999, 431, 648, 101, 111, 333, 5000, 2, 665, 9, 518, 222, 5, 20, 1]
list_seed = [123, 1000, 14, 0, 65, 456, 321, 11, 56, 100]

n_repeats, n_folds, n_estimators = 10, 5, 100

# --- MAIN LOOP ---
if __name__ == '__main__':
    for classifier_name, base_clf in base_classifiers.items():
        print(f"\n==========================\n🔍 Base Classifier: {classifier_name}\n==========================")
        for dataset in list_datasets:
            try:
                # path = "/content/drive/MyDrive/Colab Notebooks/DiverseBagging/"
                # df = pd.read_csv(path + dataset)
                df = pd.read_csv(dataset)
                X, y = df.iloc[:, :-1].values, df.iloc[:, -1].values

                # Handle non-numeric features
                # if not np.issubdtype(X.dtype, np.number):
                #     print(f"   Dataset {dataset} contains non-numeric features. Applying simple handling.")
                #     X = pd.DataFrame(X).apply(pd.to_numeric, errors='coerce').values
                #     col_mean = np.nanmean(X, axis=0)
                #     inds = np.where(np.isnan(X))
                #     X[inds] = np.take(col_mean, inds[1])

                y_encoded = LabelEncoder().fit_transform(y)
                # X_aug, y_aug = duplicate_minority_classes(X, y_encoded)
                n_classes = len(np.unique(y_encoded))
                n_features = X.shape[1]

                avacc_list, mgm_list, cba_list, f1_list, mauc_list, qstat_list = [], [], [], [], [], []
                recall_per_class_all_repeats = np.zeros((n_repeats, n_classes))
                # BARU: List untuk menyimpan waktu
                train_times_all, test_times_all = [], []

                for repeat in range(n_repeats):
                    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=list_seed[repeat])
                    recall_per_class_fold_sum = np.zeros(n_classes)

                    for train_idx, test_idx in skf.split(X, y_encoded):
                        X_train, X_test = X[train_idx], X[test_idx]
                        y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

                        models, class_weight_list = [], []
                        # BARU: Inisialisasi total waktu training fold ini
                        current_fold_train_time = 0

                        class_weights_train = calculate_class_weights(y_train)

                        # max_weight = max(class_weights_train.values()) if class_weights_train else 1
                        # normalized_weights_train = {key: value / max_weight for key, value in class_weights_train.items()}
                        # class_weights_train = normalized_weights_train

                        # Tentukan max_depth STATIS (limit, bukan randomisasi)
                        if (n_features / n_classes > 1.5):
                            max_depth = (int)(n_features)
                        else:
                            max_depth = (int)(n_features * 1.5)
                            
                        # ratio = n_features / n_classes
                        # if (ratio > 2.0):
                        #     max_depth = n_features
                        # elif (ratio > 1.0):
                        #     max_depth = (int)(n_features * 1.5)
                        # else:
                        #     max_depth = (int)(n_features * 2.0)                         

                        # --- Training Phase ---
                        start_fold_train = time.time() # Mulai catat waktu training
                        for est in range(n_estimators):
                            X_bs, y_bs = custom_bootstrap(X_train, y_train)
                            if len(np.unique(y_bs)) < n_classes or len(y_bs) == 0:
                                continue

                            class_weights = calculate_class_weights(y_bs)
                            # class_weights = class_weights_train

                            # normalisasi jika nilai weight
                            # max_weight = max(class_weights.values()) if class_weights else 1
                            # normalized_weights = {key: value / max_weight for key, value in class_weights.items()}
                            # class_weights = normalized_weights

                            # cek jika distribusi tiap kelas seimbang
                            # if np.isclose(np.prod(list(class_weights.values())), 1.0, atol=0.1, rtol=0):
                            #     # class_weights = class_weights_train
                            #     common_classes = class_weights.keys() & class_weights_train.keys()
                            #     class_weights = {
                            #         cls: min(class_weights[cls], class_weights_train[cls])
                            #         for cls in common_classes
                            #     }

                            # class_weights = {cls: min(w, 1.0) for cls, w in class_weights.items()}

                            common_classes = class_weights.keys() & class_weights_train.keys()

                            class_weights = {
                                cls: min(class_weights[cls], class_weights_train[cls])
                                for cls in common_classes
                            }

                            # class_weights = {
                            #     cls: (0.5 * class_weights[cls] + 0.5 * class_weights_train[cls])
                            #     for cls in common_classes
                            # }

                            # max_weight = min(class_weights.values()) if class_weights else 1
                            # normalized_weights = {key: value / max_weight for key, value in class_weights.items()}
                            # class_weights = normalized_weights

                            if (est <= (int)(0.5 * n_estimators)):
                                ros = RandomOverSampler(sampling_strategy='auto', random_state=None)
                                X_bs, y_bs = ros.fit_resample(X_bs, y_bs)
                            # else:
                            #     rus = RandomUnderSampler(sampling_strategy='auto', random_state=None)
                            #     X_bs, y_bs = rus.fit_resample(X_bs, y_bs)

                            # max_features = np.random.uniform(0.25, 0.75)
                            # model = DecisionTreeClassifier(max_features='sqrt')
                            model = DecisionTreeClassifier(max_depth=None)
                            # model = DecisionTreeClassifier()

                            # BARU: Catat waktu fit per model
                            start_fit = time.time()
                            try: # Tambah try-except jika fit gagal krn data bootstrap aneh
                                model.fit(X_bs, y_bs)
                                # Cek apakah model punya n_classes_ (penting untuk voting)
                                if not hasattr(model, 'n_classes_'):
                                     model.n_classes_ = n_classes # Set manual jika tidak ada
                                models.append(model)
                                class_weight_list.append(class_weights)
                            except ValueError:
                                # print("Warning: Skipping a model due to ValueError during fit (likely bootstrap issue).")
                                pass # Lewati model ini jika gagal fit
                            end_fit = time.time()
                            current_fold_train_time += (end_fit - start_fit)

                        # BARU: Simpan total waktu training fold ini
                        train_times_all.append(current_fold_train_time)
                        # --- Akhir Training Phase ---

                        # --- Testing Phase ---
                        # BARU: Catat waktu testing
                        start_test = time.time()
                        y_proba = soft_voting_with_weights(models, X_test, class_weight_list, return_proba=True)
                        y_pred = np.argmax(y_proba, axis=1)
                        end_test = time.time()
                        test_times_all.append((end_test - start_test) / len(y_test))
                        # --- Akhir Testing Phase ---

                        # Hitung metrik
                        avacc, mgm, mauc = calculate_metrics(y_test, y_pred, y_proba, n_classes)
#                             avacc, mgm, cba, f1, recall_pc = calculate_metrics(y_test, y_pred, n_classes)
#                             try:
#                                 # Perlu y_test one-hot untuk MAUC multi-class
#                                 if n_classes > 1:
#                                     y_test_one_hot = np.eye(n_classes)[y_test]
#                                     # Pastikan y_proba shape [samples, classes]
#                                     if y_proba.shape[1] == n_classes:
#                                         # mauc = roc_auc_score(y_test_one_hot, y_proba, multi_class="ovr", average="macro")
#                                     else: # Jika shape tidak cocok, beri NaN
#                                           # print(f"Warning: MAUC shape mismatch. y_test_one_hot: {y_test_one_hot.shape}, y_proba: {y_proba.shape}")
#                                         mauc = np.nan
#                                 else: # Binary atau single class
#                                      mauc = roc_auc_score(y_test, y_proba[:, 1]) if n_classes==2 and y_proba.shape[1]==2 else np.nan
#                             except Exception as e:
#                                 # print(f"Warning: MAUC calculation error: {e}")
#                                 mauc = np.nan

                        # Q-Statistik (Pastikan model_preds dibuat dari models yang valid)
                        if models: # Hanya hitung jika ada model
                            model_preds = [m.predict(X_test) for m in models]
                            q_stat = calculate_q_statistic(y_test, model_preds)
                        else:
                            q_stat = 0.0 # Default jika tidak ada model

                        avacc_list.append(avacc); mgm_list.append(mgm); #cba_list.append(cba)
                        #f1_list.append(f1); 
                        mauc_list.append(mauc); # qstat_list.append(q_stat)
                        # recall_per_class_fold_sum += recall_pc

                    # recall_per_class_all_repeats[repeat] = recall_per_class_fold_sum / n_folds

                # --- Menyimpan Hasil Akhir ---
                result = {
#                         'Classifier': classifier_name, 
                    'Dataset': dataset,
                    'AvAcc': np.mean(avacc_list), 'AvAcc_std': np.std(avacc_list),
                    'mGM': np.mean(mgm_list), 'mGM_std': np.std(mgm_list),
#                         'CBA': np.mean(cba_list), 'CBA_std': np.std(cba_list),
#                         'F1': np.mean(f1_list), 'F1_std': np.std(f1_list),
                    'MAUC': np.nanmean(mauc_list), 'MAUC_std': np.nanstd(mauc_list),
                    'Q-Statistic': np.mean(qstat_list), 'Q-Statistic_std': np.std(qstat_list),
                    # BARU: Tambahkan statistik waktu
                    'TrainTime_mean': np.mean(train_times_all), 'TrainTime_std': np.std(train_times_all),
                    'TestTime_mean': np.mean(test_times_all), 'TestTime_std': np.std(test_times_all)
                }

#                     mean_recall_per_class = np.mean(recall_per_class_all_repeats, axis=0)
#                     std_recall_per_class = np.std(recall_per_class_all_repeats, axis=0)
#                     for i in range(n_classes):
#                         result[f'Recall_Class_{i}'] = mean_recall_per_class[i]
#                         result[f'Recall_Class_{i}_std'] = std_recall_per_class[i]

                summary_results.append(result)

                # BARU: Update print untuk menyertakan waktu
                print(f"\nDataset: {dataset}")
                print(f"  AvAcc      : {result['AvAcc']:.4f} ± {result['AvAcc_std']:.4f}")
                print(f"  mGM        : {result['mGM']:.4f} ± {result['mGM_std']:.4f}")
#                     print(f"  CBA        : {result['CBA']:.4f} ± {result['CBA_std']:.4f}")
#                     print(f"  F1         : {result['F1']:.4f} ± {result['F1_std']:.4f}")
                print(f"  MAUC       : {result['MAUC']:.4f} ± {result['MAUC_std']:.4f}")
                print(f"  Q-Statistic: {result['Q-Statistic']:.4f} ± {result['Q-Statistic_std']:.4f}")
                print(f"  Train Time : {result['TrainTime_mean']:.4f} ± {result['TrainTime_std']:.4f} s")
                print(f"  Test Time  : {result['TestTime_mean']:.4f} ± {result['TestTime_std']:.4f} s")

            except FileNotFoundError:
                 print(f"❌ Gagal: File '{path + dataset}' tidak ditemukan.")
            except Exception as e:
                print(f"❌ Gagal memproses {dataset} pada {classifier_name}: {e}")
                import traceback
                traceback.print_exc() # Cetak detail error

    # Save results
    if summary_results:
        summary_df = pd.DataFrame(summary_results)
        output_filename = f"CWBagging_v1_threshold_{threshold}_timed.csv" # Nama file diubah
        summary_df.to_csv(output_filename, index=False)
        print(f"\n✅ Ringkasan disimpan ke {output_filename}")
    else:
        print("\n❌ Tidak ada hasil yang disimpan karena semua dataset gagal diproses.")


🔍 Base Classifier: DecisionTree

Dataset: winequality-red.csv
  AvAcc      : 0.4471 ± 0.0720
  mGM        : 0.2906 ± 0.2211
  MAUC       : 0.7965 ± 0.0329
  Q-Statistic: nan ± nan
  Train Time : 0.8021 ± 0.0461 s
  Test Time  : 0.0002 ± 0.0000 s

Dataset: yeast.csv
  AvAcc      : 0.5724 ± 0.0333
  mGM        : 0.3869 ± 0.2318
  MAUC       : 0.8556 ± 0.0193
  Q-Statistic: nan ± nan
  Train Time : 0.4992 ± 0.0417 s
  Test Time  : 0.0002 ± 0.0001 s

Dataset: glass.csv
  AvAcc      : 0.7808 ± 0.0501
  mGM        : 0.7389 ± 0.0583
  MAUC       : 0.9310 ± 0.0228
  Q-Statistic: nan ± nan
  Train Time : 0.1345 ± 0.0219 s
  Test Time  : 0.0005 ± 0.0001 s

Dataset: ecoli 5-fold.csv
  AvAcc      : 0.8344 ± 0.0445
  mGM        : 0.8205 ± 0.0517
  MAUC       : 0.9698 ± 0.0131
  Q-Statistic: nan ± nan
  Train Time : 0.1342 ± 0.0200 s
  Test Time  : 0.0004 ± 0.0001 s

Dataset: dermatology.csv
  AvAcc      : 0.9731 ± 0.0165
  mGM        : 0.9720 ± 0.0172
  MAUC       : 0.9980 ± 0.0022
  Q-Statistic: 

In [ ]:
# CWBagging dengan penghitung kedalaman pohon -27 Oktober 2025-

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from collections import Counter
import random
import warnings
import time

warnings.filterwarnings('ignore')

# --- SEMUA FUNGSI PEMBANTU ---
# (Tidak ada perubahan di semua fungsi pembantu)

def calculate_class_weights(y):
    n_samples = len(y)
    n_classes = len(set(y))
    class_counts = Counter(y)
    class_weights = {cls: n_samples / (n_classes * count) for cls, count in class_counts.items()}
    return class_weights

def duplicate_minority_classes(X, y, min_count=5):
    unique_classes, counts = np.unique(y, return_counts=True)
    new_X, new_y = X.copy(), y.copy()
    min_required = max(min_count, 5) # Pastikan minimal 5 untuk 5-fold CV

    for cls, count in zip(unique_classes, counts):
        if count < min_required:
            class_indices = np.where(y == cls)[0]
            n_duplicates = min_required - count
            if len(class_indices) > 0:
                duplicates = np.random.choice(class_indices, size=n_duplicates, replace=True)
                new_X = np.vstack([new_X, X[duplicates]])
                new_y = np.hstack([new_y, y[duplicates]])
    return new_X, new_y

def calculate_metrics(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
    recall_per_class = np.diag(cm) / np.sum(cm, axis=1, where=np.sum(cm, axis=1)!=0)
    avacc = np.mean(recall_per_class)
    mgm = np.prod(recall_per_class) ** (1 / n_classes) if n_classes > 0 else 0.0
    cba = np.mean([
        cm[i, i] / max(np.sum(cm[i, :]), np.sum(cm[:, i]))
        for i in range(n_classes) if max(np.sum(cm[i, :]), np.sum(cm[:, i])) > 0
    ]) if n_classes > 0 else 0.0
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return avacc, mgm, cba, macro_f1, recall_per_class

def custom_bootstrap(X, y):
    unique_classes = np.unique(y)
    class_counts = {cls: len(np.where(y == cls)[0]) for cls in unique_classes}
    largest_class_size = class_counts[max(class_counts, key=class_counts.get)] if class_counts else 1
    smallest_class_size = class_counts[min(class_counts, key=class_counts.get)] if class_counts else 1
    threshold = (largest_class_size) // 2
#     threshold = largest_class_size
#     if threshold <= 2: threshold = max(3, largest_class_size) # Pastikan threshold valid
    m = np.random.randint(2, threshold) if threshold > 2 else 2 # <--- Modifikasi kecil untuk handle threshold=2

    indices = np.array([], dtype=int)
    for cls in unique_classes:
        class_indices = np.where(y == cls)[0]
        class_size = len(class_indices)
        if not class_indices.any(): continue # Skip jika tidak ada index
        
        # Tentukan ukuran sample, pastikan tidak 0
        current_m = m if class_size > m else class_size
        if current_m == 0: continue # Skip jika ukuran sample 0
        
        sampled_indices = np.random.choice(class_indices, size=current_m, replace=True)
        indices = np.concatenate([indices, sampled_indices])
    return X[indices], y[indices]

def soft_voting_with_weights(models, X, class_weight_list, return_proba=False):
    if not models: # Handle jika list model kosong
        # Fallback: prediksi random jika tidak ada model
        n_samples = X.shape[0]
        # Coba tebak n_classes dari class_weight_list
        n_classes_est = 0
        if class_weight_list:
            n_classes_est = len(class_weight_list[0].keys())
        if n_classes_est == 0: n_classes_est = 2 # Asumsi default

        if return_proba:
            return np.ones((n_samples, n_classes_est)) / n_classes_est
        else:
            return np.random.randint(0, n_classes_est, size=n_samples)


    n_samples = X.shape[0]
    n_classes = models[0].n_classes_
    weighted_predictions = np.zeros((n_samples, n_classes))

    for i, model in enumerate(models):
        probas = model.predict_proba(X)
        
        # Handle jika probas tidak lengkap (model tidak melihat semua kelas saat fit)
        if probas.shape[1] < n_classes:
            temp_probas = np.zeros((n_samples, n_classes))
            try:
                # Gunakan model.classes_ untuk mapping yang benar
                temp_probas[:, model.classes_] = probas
            except Exception: 
                 # Fallback jika model.classes_ tidak sesuai
                temp_probas[:, :probas.shape[1]] = probas # Asumsi kelas 0..k-1
            probas = temp_probas

        preds = np.argmax(probas, axis=1)
        confidences = np.max(probas, axis=1)
        class_weights = class_weight_list[i]

        for j in range(n_samples):
            pred_class = preds[j]
            if pred_class in class_weights:
                weighted_predictions[j, pred_class] += confidences[j] * class_weights[pred_class]

    if return_proba:
        row_sums = weighted_predictions.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        return weighted_predictions / row_sums

    final_predictions = np.argmax(weighted_predictions, axis=1)
    return final_predictions

def calculate_q_statistic(y_true, predictions):
    if not predictions or len(predictions) < 2: return 0.0 # Handle jika tidak cukup model
    n_models = len(predictions)
    q_values = []
    y_true = np.array(y_true).flatten()

    for i in range(n_models):
        for j in range(i + 1, n_models):
            pred_i = np.array(predictions[i]).flatten()
            pred_j = np.array(predictions[j]).flatten()
            if pred_i.shape != y_true.shape or pred_j.shape != y_true.shape:
                continue 

            correct_i = (pred_i == y_true)
            correct_j = (pred_j == y_true)

            N11 = np.sum(correct_i & correct_j)
            N00 = np.sum(~correct_i & ~correct_j)
            N10 = np.sum(correct_i & ~correct_j)
            N01 = np.sum(~correct_i & correct_j)

            denominator = (N11 * N00 + N10 * N01)
            q = (N11 * N00 - N10 * N01) / denominator if denominator != 0 else 0
            q_values.append(q)

    return np.mean(q_values) if q_values else 0.0

# --- PENGATURAN EKSPERIMEN ---

base_classifiers = {
    'DecisionTree': DecisionTreeClassifier(),
}
list_datasets = ['winequality-red.csv', 'yeast.csv', 'glass.csv', 'ecoli.csv', 'dermatology.csv', 'solar-flare.csv',
                 'contraceptive.csv', 'balance-scale.csv', 'led7digit.csv', 'lymphography.csv', 'hayes-roth.csv',
                 'cleveland.csv', 'new-thyroid.csv', 'page-blocks.csv', 'wine.csv', 'thyroid.csv', 'zoo.csv', 'shuttle.csv']
summary_results = []
list_threshold = [0]
list_seed = [123, 1000, 14, 0, 65, 456, 321, 11, 56, 100]
# path = "/content/drive/MyDrive/DiverseBagging/" # Sesuaikan path

# --- MAIN LOOP ---
if __name__ == '__main__':
    for threshold in list_threshold:
        for classifier_name, base_clf in base_classifiers.items():
            print(f"\n==========================\n🔍 Base Classifier: {classifier_name}\n==========================")
            for dataset in list_datasets:
                try:
                    df = pd.read_csv(dataset)
                    X, y = df.iloc[:, :-1].values, df.iloc[:, -1].values
                    
                    # Handle non-numeric features
                    if not np.issubdtype(X.dtype, np.number):
                        print(f"   Dataset {dataset} contains non-numeric features. Applying simple handling.")
                        X = pd.DataFrame(X).apply(pd.to_numeric, errors='coerce').values
                        col_mean = np.nanmean(X, axis=0)
                        inds = np.where(np.isnan(X))
                        X[inds] = np.take(col_mean, inds[1])

                    y_encoded = LabelEncoder().fit_transform(y)
                    X_aug, y_aug = duplicate_minority_classes(X, y_encoded)
                    n_classes = len(np.unique(y_encoded))
                    n_features = X.shape[1]

                    n_repeats, n_folds, n_estimators = 10, 5, 100
                    avacc_list, mgm_list, cba_list, f1_list, mauc_list, qstat_list = [], [], [], [], [], []
                    recall_per_class_all_repeats = np.zeros((n_repeats, n_classes))
                    train_times_all, test_times_all = [], []
                    
                    # <--- BARU: List untuk menyimpan statistik kedalaman dari semua fold
                    fold_avg_depths = []
                    fold_max_depths = []
                    fold_min_depths = []


                    for repeat in range(n_repeats):
                        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=list_seed[repeat])
                        recall_per_class_fold_sum = np.zeros(n_classes)

                        for train_idx, test_idx in skf.split(X_aug, y_aug):
                            X_train, X_test = X_aug[train_idx], X_aug[test_idx]
                            y_train, y_test = y_aug[train_idx], y_aug[test_idx]
                            
                            models, class_weight_list = [], []
                            current_fold_train_time = 0
                            
                            class_weights_train = calculate_class_weights(y_train)
                            
                            # --- Training Phase ---
                            start_fold_train = time.time() # Mulai catat waktu training
                            for _ in range(n_estimators):
                                X_bs, y_bs = custom_bootstrap(X_train, y_train)
                                if len(np.unique(y_bs)) < n_classes or len(y_bs) == 0:
                                    continue
                                
                                class_weights = calculate_class_weights(y_bs)
                                
                                # Normalisasi class_weights
                                max_weight = max(class_weights.values()) if class_weights else 1
                                if max_weight == 0: max_weight = 1 # Hindari division by zero
                                normalized_weights = {key: value / max_weight for key, value in class_weights.items()}
                                class_weights = normalized_weights
                                
                                # Tentukan max_depth (jika diaktifkan)
                                # if (n_features / n_classes >= 2):
                                #    max_depth = np.random.randint((n_features // 2), (int)(n_features * 1.5))
                                #else:
                                #    max_depth = (int)(n_features * 1.5) 
                                
                                # Anda HARUS mengganti base_clf di sini jika ingin pakai classifier lain
                                # model = DecisionTreeClassifier(max_depth=max_depth) # Jika pakai aturan max_depth
                                model = DecisionTreeClassifier(max_depth=None) # Sesuai kode asli Anda
                                
                                start_fit = time.time()
                                try: 
                                    model.fit(X_bs, y_bs)
                                    if not hasattr(model, 'n_classes_'):
                                          model.n_classes_ = n_classes 
                                    # Pastikan model.classes_ ada, jika tidak, set manual
                                    if not hasattr(model, 'classes_'):
                                          model.classes_ = np.unique(y_bs)
                                          
                                    models.append(model)
                                    class_weight_list.append(class_weights)
                                except ValueError:
                                    pass 
                                end_fit = time.time()
                                current_fold_train_time += (end_fit - start_fit)

                            train_times_all.append(current_fold_train_time)
                            # --- Akhir Training Phase ---

                            # <--- BARU: Hitung statistik kedalaman dari 100 pohon di fold ini
                            if models: # Pastikan list 'models' tidak kosong
                                # 1. Ekstrak kedalaman setiap pohon
                                #    (Hanya berfungsi jika base classifier adalah DecisionTree)
                                if classifier_name == 'DecisionTree':
                                    depths_in_fold = [tree.tree_.max_depth for tree in models]
                                    
                                    # 2. Hitung statistik dan simpan
                                    fold_avg_depths.append(np.mean(depths_in_fold))
                                    fold_max_depths.append(np.max(depths_in_fold))
                                    fold_min_depths.append(np.min(depths_in_fold))
                                else:
                                    # Jika bukan DecisionTree, catat sebagai NaN atau 0
                                    fold_avg_depths.append(np.nan)
                                    fold_max_depths.append(np.nan)
                                    fold_min_depths.append(np.nan)
                            else:
                                # Jika tidak ada model yg valid, catat sebagai 0
                                fold_avg_depths.append(0)
                                fold_max_depths.append(0)
                                fold_min_depths.append(0)


                            # --- Testing Phase ---
                            start_test = time.time()
                            y_proba = soft_voting_with_weights(models, X_test, class_weight_list, return_proba=True)
                            y_pred = np.argmax(y_proba, axis=1)
                            end_test = time.time()
                            test_times_all.append((end_test - start_test) / len(y_test)) # Waktu per instance
                            # --- Akhir Testing Phase ---

                            # Hitung metrik
                            avacc, mgm, cba, f1, recall_pc = calculate_metrics(y_test, y_pred, n_classes)
                            try:
                                if n_classes > 1:
                                    y_test_one_hot = np.eye(n_classes)[y_test]
                                    if y_proba.shape[1] == n_classes:
                                         mauc = roc_auc_score(y_test_one_hot, y_proba, multi_class="ovr", average="macro")
                                    else: 
                                         mauc = np.nan
                                else: 
                                     mauc = roc_auc_score(y_test, y_proba[:, 1]) if n_classes==2 and y_proba.shape[1]==2 else np.nan
                            except Exception as e:
                                mauc = np.nan

                            # Q-Statistik
                            if models: 
                                model_preds = [m.predict(X_test) for m in models]
                                q_stat = calculate_q_statistic(y_test, model_preds)
                            else:
                                q_stat = 0.0 

                            avacc_list.append(avacc); mgm_list.append(mgm); cba_list.append(cba)
                            f1_list.append(f1); mauc_list.append(mauc); qstat_list.append(q_stat)
                            recall_per_class_fold_sum += recall_pc
                            
                        recall_per_class_all_repeats[repeat] = recall_per_class_fold_sum / n_folds

                    # --- Menyimpan Hasil Akhir ---
                    result = {
                        'Classifier': classifier_name, 'Dataset': dataset,
                        'AvAcc': np.mean(avacc_list), 'AvAcc_std': np.std(avacc_list),
                        'mGM': np.mean(mgm_list), 'mGM_std': np.std(mgm_list),
                        'CBA': np.mean(cba_list), 'CBA_std': np.std(cba_list),
                        'F1': np.mean(f1_list), 'F1_std': np.std(f1_list),
                        'MAUC': np.nanmean(mauc_list), 'MAUC_std': np.nanstd(mauc_list),
                        'Q-Statistic': np.mean(qstat_list), 'Q-Statistic_std': np.std(qstat_list),
                        'TrainTime_mean': np.mean(train_times_all), 'TrainTime_std': np.std(train_times_all),
                        'TestTime_mean': np.mean(test_times_all), 'TestTime_std': np.std(test_times_all),
                        
                        # <--- BARU: Tambahkan statistik kedalaman (rata-rata dari semua fold)
                        'Avg_Depth_Mean': np.nanmean(fold_avg_depths), 'Avg_Depth_Std': np.nanstd(fold_avg_depths),
                        'Max_Depth_Mean': np.nanmean(fold_max_depths), 'Max_Depth_Std': np.nanstd(fold_max_depths),
                        'Min_Depth_Mean': np.nanmean(fold_min_depths), 'Min_Depth_Std': np.nanstd(fold_min_depths),
                    }
                    
                    mean_recall_per_class = np.mean(recall_per_class_all_repeats, axis=0)
                    std_recall_per_class = np.std(recall_per_class_all_repeats, axis=0)
                    for i in range(n_classes):
                        result[f'Recall_Class_{i}'] = mean_recall_per_class[i]
                        result[f'Recall_Class_{i}_std'] = std_recall_per_class[i]
                    
                    summary_results.append(result)

                    # <--- BARU: Update print untuk menyertakan kedalaman
                    print(f"\nDataset: {dataset}")
                    print(f"  AvAcc        : {result['AvAcc']:.4f} ± {result['AvAcc_std']:.4f}")
                    print(f"  mGM          : {result['mGM']:.4f} ± {result['mGM_std']:.4f}")
                    print(f"  CBA          : {result['CBA']:.4f} ± {result['CBA_std']:.4f}")
                    print(f"  F1           : {result['F1']:.4f} ± {result['F1_std']:.4f}")
                    print(f"  MAUC         : {result['MAUC']:.4f} ± {result['MAUC_std']:.4f}")
                    print(f"  Q-Statistic  : {result['Q-Statistic']:.4f} ± {result['Q-Statistic_std']:.4f}")
                    print(f"  Train Time   : {result['TrainTime_mean']:.4f} ± {result['TrainTime_std']:.4f} s")
                    print(f"  Test Time    : {result['TestTime_mean']:.4f} ± {result['TestTime_std']:.4f} s")
                    print(f"  Depth (Avg)  : {result['Avg_Depth_Mean']:.2f} ± {result['Avg_Depth_Std']:.2f}")
                    print(f"  Depth (Max)  : {result['Max_Depth_Mean']:.2f} ± {result['Max_Depth_Std']:.2f}")
                    print(f"  Depth (Min)  : {result['Min_Depth_Mean']:.2f} ± {result['Min_Depth_Std']:.2f}")

                except FileNotFoundError:
                        print(f"❌ Gagal: File '{dataset}' tidak ditemukan.") # Path dihapus
                except Exception as e:
                    print(f"❌ Gagal memproses {dataset} pada {classifier_name}: {e}")
                    import traceback
                    traceback.print_exc() # Cetak detail error

    # Save results
    if summary_results:
        summary_df = pd.DataFrame(summary_results)
        # Nama file diubah untuk mencerminkan pelacakan kedalaman
        output_filename = f"CWBagging_v1_threshold_{threshold}_depth_analysis_01.csv" 
        summary_df.to_csv(output_filename, index=False)
        print(f"\n✅ Ringkasan disimpan ke {output_filename}")
    else:
        print("\n❌ Tidak ada hasil yang disimpan karena semua dataset gagal diproses.")

In [ ]:
# Kode CWBagging dengan variasi base classifier

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
from collections import Counter
import random
import warnings
import time

warnings.filterwarnings('ignore')

# Fungsi untuk menghitung class weights
def calculate_class_weights(y):
    n_samples = len(y)
    n_classes = len(set(y))
    class_counts = Counter(y)
    class_weights = {cls: n_samples / (n_classes * count) for cls, count in class_counts.items()}
    return class_weights

# Duplikasi data pada kelas dengan jumlah data < 5
def duplicate_minority_classes(X, y, min_count=5):
    unique_classes, counts = np.unique(y, return_counts=True)
    new_X, new_y = X.copy(), y.copy()
    
    for cls, count in zip(unique_classes, counts):
        if count < min_count:
            class_indices = np.where(y == cls)[0]
            n_duplicates = min_count - count
            duplicates = np.random.choice(class_indices, size=n_duplicates, replace=True)
            new_X = np.vstack([new_X, X[duplicates]])
            new_y = np.hstack([new_y, y[duplicates]])
    
    return new_X, new_y

# Fungsi untuk menghitung metrik AvAcc, mGM, CBA, Macro F1
def calculate_metrics(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
    recall_per_class = np.diag(cm) / np.sum(cm, axis=1)
    
    avacc = np.mean(recall_per_class)
    mgm = np.prod(recall_per_class) ** (1 / n_classes)
    cba = np.mean([
        cm[i, i] / max(np.sum(cm[i, :]), np.sum(cm[:, i]))
        for i in range(n_classes)
    ])
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    
    return avacc, mgm, cba, macro_f1, recall_per_class

# Custom bootstrap sampling
def custom_bootstrap(X, y):
    unique_classes = np.unique(y)
    class_counts = {cls: len(np.where(y == cls)[0]) for cls in unique_classes}
    largest_class = max(class_counts, key=class_counts.get)
    largest_class_size = class_counts[largest_class]
    sampling_percentage = np.random.uniform(0.1, 1.0)
    # sampling_percentage = np.random.uniform(0.1, 0.5)
    m = int(sampling_percentage * largest_class_size)
    
    indices = np.array([], dtype=int)
    for cls in unique_classes:
        class_indices = np.where(y == cls)[0]
        class_size = len(class_indices)
        if class_size > m:
            sampled_indices = np.random.choice(class_indices, size=m, replace=True)
        else:
            sampled_indices = np.random.choice(class_indices, size=class_size, replace=True)
        indices = np.concatenate([indices, sampled_indices])
    
    return X[indices], y[indices]

# def custom_bootstrap(X, y, random_state=None):
#     if random_state is not None:
#         np.random.seed(random_state)
#         random.seed(random_state)

#     # Hitung jumlah data per kelas
#     class_counts = Counter(y)

#     # Rata-rata jumlah data di semua kelas
#     avg_all_classes = int(np.mean(list(class_counts.values())))
#     size = random.randint(2, avg_all_classes)

#     # Buat indeks data per kelas
#     class_indices = {cls: np.where(y == cls)[0] for cls in np.unique(y)}

#     indices = []
#     for cls, idxs in class_indices.items():
#         if len(idxs) > size:
#             sampled = np.random.choice(idxs, size=size, replace=False)
#         else:
#             sampled = np.random.choice(idxs, size=len(idxs), replace=True)
#         indices.extend(sampled)

#     np.random.shuffle(indices)
#     return X[indices], y[indices]


# Soft voting with weights
def soft_voting_with_weights(models, X, class_weight_list, threshold=0.6):
    n_samples = X.shape[0]
    n_classes = len(class_weight_list[0])
    weighted_predictions = np.zeros((n_samples, n_classes))
    
    for i, model in enumerate(models):
        probas = model.predict_proba(X)
        preds = np.argmax(probas, axis=1)
        confidences = np.max(probas, axis=1)
        class_weights = class_weight_list[i]
        
        for j in range(n_samples):
            if confidences[j] >= threshold:
                pred_class = preds[j]
                weighted_predictions[j, pred_class] += confidences[j] * class_weights[pred_class]
    
    final_predictions = np.argmax(weighted_predictions, axis=1)
    return final_predictions

# Base classifier dictionary
base_classifiers = {
    'DecisionTree': DecisionTreeClassifier(),
#     'KNN': KNeighborsClassifier(n_neighbors=5),
#     'NaiveBayes': GaussianNB(),
#     'LR': LogisticRegression(max_iter=200, random_state=None)
}

# Dataset list
list_datasets = ['winequality-red.csv', 'yeast.csv', 'glass.csv', 'ecoli.csv', 'dermatology.csv', 'solar-flare.csv', 
                 'contraceptive.csv', 'balance-scale.csv', 'led7digit.csv', 'lymphography.csv', 'hayes-roth.csv',
                 'cleveland.csv', 'new-thyroid.csv', 'page-blocks.csv', 'wine.csv', 'thyroid.csv', 'zoo.csv', 'shuttle.csv']

# Main Loop
summary_results = []

for classifier_name, base_clf in base_classifiers.items():
    print(f"\n==========================")
    print(f"🔍 Base Classifier: {classifier_name}")
    print(f"==========================")

    for dataset in list_datasets:
        try:
            df = pd.read_csv(dataset)
            X = df.iloc[:, :-1].values
            y = df.iloc[:, -1].values
            y_encoded = LabelEncoder().fit_transform(y)
            X_aug, y_aug = duplicate_minority_classes(X, y_encoded)
            n_classes = len(np.unique(y_encoded))
            
            n_repeats, n_folds, n_estimators = 20, 5, 100
            avacc_list, mgm_list, cba_list, f1_list = [], [], [], []
            recall_per_class_all = np.zeros((n_repeats, n_classes))
            
            for repeat in range(n_repeats):
                skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=None)
                for train_idx, test_idx in skf.split(X_aug, y_aug):
                    X_train, X_test = X_aug[train_idx], X_aug[test_idx]
                    y_train, y_test = y_aug[train_idx], y_aug[test_idx]
                    
                    models, class_weight_list = [], []
                    for _ in range(n_estimators):
                        X_bs, y_bs = custom_bootstrap(X_train, y_train)
                        class_weights = calculate_class_weights(y_bs)
                        model = base_clf.__class__(**base_clf.get_params())
                        model.fit(X_bs, y_bs)
                        models.append(model)
                        class_weight_list.append(class_weights)
                    
                    y_pred = soft_voting_with_weights(models, X_test, class_weight_list)
                    avacc, mgm, cba, f1, recall_pc = calculate_metrics(y_test, y_pred, n_classes)
                    avacc_list.append(avacc)
                    mgm_list.append(mgm)
                    cba_list.append(cba)
                    f1_list.append(f1)
                    recall_per_class_all[repeat] += recall_pc / n_folds
            
            result = {
                'Classifier': classifier_name,
                'Dataset': dataset,
                'AvAcc': np.mean(avacc_list),
                'mGM': np.mean(mgm_list),
                'CBA': np.mean(cba_list),
                'F1': np.mean(f1_list)
            }
            for i in range(n_classes):
                result[f'Recall_Class_{i}'] = np.mean(recall_per_class_all[:, i])
            
            summary_results.append(result)
            
            print(f"\nDataset: {dataset}")
            print(f"  AvAcc: {result['AvAcc']:.4f}")
            print(f"  mGM  : {result['mGM']:.4f}")
            print(f"  CBA  : {result['CBA']:.4f}")
            print(f"  F1   : {result['F1']:.4f}")
            # print recall per class if you want
            # for i in range(n_classes):
            #     print(f"    Recall Class {i}: {result[f'Recall_Class_{i}']:.4f}")

        except Exception as e:
            print(f"❌ Gagal memproses {dataset} pada {classifier_name}: {e}")

# Save results to CSV
summary_df = pd.DataFrame(summary_results)
summary_df.to_csv("CWBagging_summary_results_threshold.csv", index=False)
print("\n✅ Ringkasan disimpan ke 'CWBagging_summary_results.csv'")
